# Physical Health Risk Model Training

**Novelle — AI-Powered Maternal Health Risk Support Platform**

This notebook trains the physical/maternal health risk prediction model using an Ensemble approach.

## Model Overview
- **Target**: Physical health risk level (LOW / MEDIUM / HIGH)
- **Input Features**: BP, blood sugar, weight, symptoms, pregnancy week
- **Algorithm**: Ensemble (XGBoost + Random Forest + Logistic Regression)
- **Explainability**: SHAP values for feature importance

---

In [3]:
# Install dependencies (run once)
# !pip install pandas numpy scikit-learn xgboost lightgbm shap imbalanced-learn matplotlib seaborn joblib

## 1. Import Libraries

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# ML imports
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, 
    f1_score, roc_auc_score, precision_score, recall_score,
    ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap
import joblib

print("✅ Libraries loaded successfully")
print(f"   XGBoost version: {xgb.__version__}")

Task was destroyed but it is pending!
task: <Task pending name='Task-175' coro=<_async_in_context.<locals>.run_in_context() done, defined at /home/linxcapture/Desktop/projects/pregency-friend/backend/venv/lib/python3.12/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-176' coro=<Kernel.shell_main() running at /home/linxcapture/Desktop/projects/pregency-friend/backend/venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /home/linxcapture/Desktop/projects/pregency-friend/backend/venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py:563]>
Task was destroyed but it is pending!
task: <Task pending name='Task-176' coro=<Kernel.shell_main() running at /home/linxcapture/Desktop/projects/pregency-friend/backend/venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]>


✅ Libraries loaded successfully
   XGBoost version: 2.0.3


## 2. Load Data

In [ ]:
# Paths
DATA_DIR = Path('../datasets')
MODEL_DIR = Path('../../backend/app/ml/models')
REPORT_DIR = Path('../reports')

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Load Maternal Health Risk Dataset
df = pd.read_csv(DATA_DIR / 'Maternal Health Risk Data Set.csv')

print(f"Records: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print(f"\nOriginal Risk Distribution:")
print(df['RiskLevel'].value_counts())
df.head()

## 3. Exploratory Data Analysis

In [ ]:
# Dataset info
print("=" * 50)
print("MATERNAL HEALTH DATASET INFO")
print("=" * 50)
print(df.info())
print("\n" + "=" * 50)
print("STATISTICAL SUMMARY")
print("=" * 50)
df.describe()

In [ ]:
# Data Cleaning: Standardize risk labels
label_map = {'low risk': 'LOW', 'mid risk': 'MEDIUM', 'high risk': 'HIGH'}
df['risk_label'] = df['RiskLevel'].map(label_map)

print("Cleaned Risk Distribution:")
print(df['risk_label'].value_counts())

# Target distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Risk label distribution
risk_counts = df['risk_label'].value_counts()
colors = {'LOW': '#4CAF50', 'MEDIUM': '#FFC107', 'HIGH': '#F44336'}
risk_counts.plot(kind='bar', ax=axes[0], color=[colors.get(x, '#666') for x in risk_counts.index])
axes[0].set_title('Maternal Health Risk Distribution')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# BP distribution by risk
for label in ['LOW', 'MEDIUM', 'HIGH']:
    if label in df['risk_label'].values:
        subset = df[df['risk_label'] == label]
        axes[1].scatter(subset['SystolicBP'], subset['DiastolicBP'], 
                       alpha=0.4, label=label, color=colors[label], s=20)
axes[1].set_title('Blood Pressure by Risk Level')
axes[1].set_xlabel('Systolic BP (mmHg)')
axes[1].set_ylabel('Diastolic BP (mmHg)')
axes[1].legend()
axes[1].axhline(y=90, color='red', linestyle='--', alpha=0.5)
axes[1].axvline(x=140, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(REPORT_DIR / 'physical_health_eda.png', dpi=150)
plt.show()

In [ ]:
# Feature correlation heatmap
numeric_cols = ['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate']
plt.figure(figsize=(8, 6))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='RdYlGn_r', center=0, fmt='.2f')
plt.title('Feature Correlation Matrix - Maternal Health')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'physical_health_correlation.png', dpi=150)
plt.show()

## 4. Feature Engineering

In [ ]:
# Feature engineering on Maternal Health Dataset
df['bp_mean'] = (df['SystolicBP'] + df['DiastolicBP']) / 2
df['bp_pulse_pressure'] = df['SystolicBP'] - df['DiastolicBP']
df['hypertension_flag'] = ((df['SystolicBP'] >= 140) | (df['DiastolicBP'] >= 90)).astype(int)
df['hyperglycemia_flag'] = (df['BS'] >= 7.8).astype(int)  # BS appears to be mmol/L
df['fever_flag'] = (df['BodyTemp'] > 99.5).astype(int)
df['tachycardia_flag'] = (df['HeartRate'] > 100).astype(int)
df['bradycardia_flag'] = (df['HeartRate'] < 60).astype(int)

# Age risk categories
df['age_risk'] = pd.cut(df['Age'], bins=[0, 18, 35, 100], labels=[1, 0, 1]).astype(int)  # Young/old = higher risk

# Blood sugar severity
df['bs_severity'] = pd.cut(df['BS'], bins=[0, 6.1, 7.8, 11.1, 100], 
                           labels=[0, 1, 2, 3]).astype(float)

print(f"Engineered features added. Total features: {len(df.columns)}")
print(f"Columns: {list(df.columns)}")
df.head()

## 5. Prepare Training Data

In [ ]:
# Define features based on Maternal Health Dataset
FEATURE_COLS = [
    # Original features
    'Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate',
    # Engineered features
    'bp_mean', 'bp_pulse_pressure', 
    'hypertension_flag', 'hyperglycemia_flag', 'fever_flag',
    'tachycardia_flag', 'bradycardia_flag', 'age_risk', 'bs_severity'
]

# Prepare X and y
X = df[FEATURE_COLS].copy()
y = df['risk_label'].copy()

# Handle missing values
X = X.fillna(X.median())

# Encode labels
label_encoder = LabelEncoder()
label_encoder.fit(['LOW', 'MEDIUM', 'HIGH'])
y_encoded = label_encoder.transform(y)

print(f"Features: {X.shape[1]}")
print(f"Samples: {X.shape[0]}")
print(f"\nClass distribution:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {cls}: {(y_encoded == i).sum()} ({(y_encoded == i).mean()*100:.1f}%)")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Train set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Handle class imbalance with SMOTE
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print(f"After SMOTE: {len(X_train_balanced)} samples")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {cls}: {(y_train_balanced == i).sum()}")

## 6. Model Training - Ensemble Approach

In [ ]:
# Individual models
print("Training individual models...\n")

# 1. XGBoost
xgb_clf = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# 2. Random Forest
rf_clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

# 3. Logistic Regression
lr_clf = LogisticRegression(
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

# Train individual models
xgb_clf.fit(X_train_balanced, y_train_balanced)
print("✅ XGBoost trained")

rf_clf.fit(X_train_balanced, y_train_balanced)
print("✅ Random Forest trained")

lr_clf.fit(X_train_balanced, y_train_balanced)
print("✅ Logistic Regression trained")

In [ ]:
# Evaluate individual models
models = {
    'XGBoost': xgb_clf,
    'Random Forest': rf_clf,
    'Logistic Regression': lr_clf
}

print("\n" + "=" * 60)
print("INDIVIDUAL MODEL PERFORMANCE")
print("=" * 60)

for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    print(f"{name:25} | Accuracy: {acc:.4f} | F1: {f1:.4f}")

In [ ]:
# Create VotingClassifier ensemble
# Note: XGBoost in VotingClassifier can be tricky; we'll use soft voting with RF and LR
ensemble = VotingClassifier(
    estimators=[
        ('rf', rf_clf),
        ('lr', lr_clf)
    ],
    voting='soft',
    n_jobs=-1
)

# Train ensemble
print("\nTraining Voting Ensemble (RF + LR)...")
ensemble.fit(X_train_balanced, y_train_balanced)
print("✅ Ensemble trained")

In [ ]:
# Compare ensemble with XGBoost standalone
y_pred_ensemble = ensemble.predict(X_test_scaled)
y_pred_xgb = xgb_clf.predict(X_test_scaled)

acc_ensemble = accuracy_score(y_test, y_pred_ensemble)
acc_xgb = accuracy_score(y_test, y_pred_xgb)
f1_ensemble = f1_score(y_test, y_pred_ensemble, average='weighted')
f1_xgb = f1_score(y_test, y_pred_xgb, average='weighted')

print("\n" + "=" * 60)
print("ENSEMBLE vs XGBOOST COMPARISON")
print("=" * 60)
print(f"Ensemble (RF+LR)         | Accuracy: {acc_ensemble:.4f} | F1: {f1_ensemble:.4f}")
print(f"XGBoost Standalone       | Accuracy: {acc_xgb:.4f} | F1: {f1_xgb:.4f}")

# Use the better performing model
if f1_xgb > f1_ensemble:
    best_model = xgb_clf
    best_model_name = 'XGBoost'
    y_pred = y_pred_xgb
else:
    best_model = ensemble
    best_model_name = 'Ensemble'
    y_pred = y_pred_ensemble

print(f"\n→ Using {best_model_name} as final model")

## 7. Model Evaluation

In [ ]:
# Get probabilities from best model
y_pred_proba = best_model.predict_proba(X_test_scaled)

# Metrics
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

# AUC-ROC
try:
    auc_roc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted')
except:
    auc_roc = 0.0

print("\n" + "=" * 50)
print("MODEL EVALUATION METRICS")
print("=" * 50)
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print(f"  AUC-ROC:   {auc_roc:.4f}")

In [ ]:
# Classification report
print("\n" + "=" * 50)
print("CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(ax=ax, cmap='Greens', values_format='d')
plt.title('Physical Health Risk - Confusion Matrix')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'physical_health_confusion_matrix.png', dpi=150)
plt.show()

## 8. Feature Importance & SHAP Analysis

In [ ]:
# Feature importance (use XGBoost for interpretability)
feature_importance = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': xgb_clf.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(feature_importance['feature'], feature_importance['importance'], color='teal')
plt.xlabel('Importance')
plt.title('XGBoost Feature Importance - Maternal Health Model')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'physical_health_feature_importance.png', dpi=150)
plt.show()

In [ ]:
# SHAP values (using XGBoost)
print("Computing SHAP values...")
explainer = shap.TreeExplainer(xgb_clf)
shap_values = explainer.shap_values(X_test_scaled[:100])

# SHAP summary plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_scaled[:100], feature_names=FEATURE_COLS,
                  class_names=label_encoder.classes_, show=False)
plt.tight_layout()
plt.savefig(REPORT_DIR / 'physical_health_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Save Model Artifacts

In [ ]:
# Save model artifacts
joblib.dump(xgb_clf, MODEL_DIR / 'physical_health_ensemble.joblib')
joblib.dump(scaler, MODEL_DIR / 'physical_health_scaler.joblib')
joblib.dump(label_encoder, MODEL_DIR / 'physical_health_label_encoder.joblib')

# Save feature columns for inference
with open(MODEL_DIR / 'physical_health_features.json', 'w') as f:
    json.dump(FEATURE_COLS, f)

print("✅ Model artifacts saved:")
print(f"   - {MODEL_DIR / 'physical_health_ensemble.joblib'}")
print(f"   - {MODEL_DIR / 'physical_health_scaler.joblib'}")
print(f"   - {MODEL_DIR / 'physical_health_label_encoder.joblib'}")
print(f"   - {MODEL_DIR / 'physical_health_features.json'}")

In [ ]:
# Save evaluation metrics
metrics = {
    'physical_health': {
        'model': best_model_name,
        'accuracy': round(accuracy, 4),
        'precision': round(precision, 4),
        'recall': round(recall, 4),
        'f1_score': round(f1, 4),
        'auc_roc': round(auc_roc, 4),
        'feature_columns': FEATURE_COLS,
        'dataset': 'Maternal Health Risk Data Set'
    }
}

# Load existing or create new
metrics_file = REPORT_DIR / 'evaluation_metrics.json'
if metrics_file.exists():
    with open(metrics_file, 'r') as f:
        all_metrics = json.load(f)
    all_metrics.update(metrics)
else:
    all_metrics = metrics

with open(metrics_file, 'w') as f:
    json.dump(all_metrics, f, indent=2)

print(f"\n✅ Metrics saved to {metrics_file}")

## 10. Model Inference Test

In [ ]:
def predict_physical_risk(age, systolic_bp, diastolic_bp, blood_sugar, body_temp, heart_rate):
    """Predict physical/maternal health risk from input features."""
    # Load artifacts
    model = joblib.load(MODEL_DIR / 'physical_health_ensemble.joblib')
    scaler_loaded = joblib.load(MODEL_DIR / 'physical_health_scaler.joblib')
    encoder = joblib.load(MODEL_DIR / 'physical_health_label_encoder.joblib')
    
    # Engineered features
    bp_mean = (systolic_bp + diastolic_bp) / 2
    bp_pulse_pressure = systolic_bp - diastolic_bp
    hypertension_flag = int(systolic_bp >= 140 or diastolic_bp >= 90)
    hyperglycemia_flag = int(blood_sugar >= 7.8)
    fever_flag = int(body_temp > 99.5)
    tachycardia_flag = int(heart_rate > 100)
    bradycardia_flag = int(heart_rate < 60)
    age_risk = 1 if (age < 18 or age > 35) else 0
    bs_severity = 0 if blood_sugar < 6.1 else (1 if blood_sugar < 7.8 else (2 if blood_sugar < 11.1 else 3))
    
    features = np.array([[
        age, systolic_bp, diastolic_bp, blood_sugar, body_temp, heart_rate,
        bp_mean, bp_pulse_pressure, 
        hypertension_flag, hyperglycemia_flag, fever_flag,
        tachycardia_flag, bradycardia_flag, age_risk, bs_severity
    ]])
    
    features_scaled = scaler_loaded.transform(features)
    prediction = model.predict(features_scaled)[0]
    probabilities = model.predict_proba(features_scaled)[0]
    
    risk_label = encoder.inverse_transform([prediction])[0]
    confidence = probabilities[prediction]
    
    return {
        'risk_level': risk_label,
        'confidence': round(confidence, 3),
        'probabilities': {cls: round(prob, 3) for cls, prob in zip(encoder.classes_, probabilities)}
    }

# Test cases
print("\n" + "=" * 50)
print("INFERENCE TEST")
print("=" * 50)

# Low risk profile
result = predict_physical_risk(age=28, systolic_bp=110, diastolic_bp=70, blood_sugar=5.5, body_temp=98.2, heart_rate=75)
print(f"\nLow Risk: Age=28, BP=110/70, BS=5.5, Temp=98.2, HR=75")
print(f"  → Prediction: {result['risk_level']} (confidence: {result['confidence']})")

# Medium risk profile
result = predict_physical_risk(age=35, systolic_bp=135, diastolic_bp=88, blood_sugar=8.5, body_temp=98.5, heart_rate=82)
print(f"\nMedium Risk: Age=35, BP=135/88, BS=8.5, Temp=98.5, HR=82")
print(f"  → Prediction: {result['risk_level']} (confidence: {result['confidence']})")

# High risk profile  
result = predict_physical_risk(age=42, systolic_bp=155, diastolic_bp=100, blood_sugar=15.0, body_temp=100.5, heart_rate=95)
print(f"\nHigh Risk: Age=42, BP=155/100, BS=15.0, Temp=100.5, HR=95")
print(f"  → Prediction: {result['risk_level']} (confidence: {result['confidence']})")

---

## Summary

✅ **Physical Health Risk Model trained successfully!**

| Metric | Value |
|--------|-------|
| Algorithm | Ensemble (XGBoost / RF+LR) |
| Features | ~30 (including engineered) |
| Target | LOW / MEDIUM / HIGH |

### Key Risk Indicators
- **Hypertension**: BP ≥ 140/90 mmHg
- **Hyperglycemia**: Fasting ≥ 126 mg/dL or Postmeal ≥ 200 mg/dL
- **Symptoms**: Edema, Bleeding, Dizziness, Cramps

### Model Artifacts Saved
- `physical_health_ensemble.joblib` — Trained model
- `physical_health_scaler.joblib` — StandardScaler
- `physical_health_label_encoder.joblib` — LabelEncoder

---

⚠️ **Disclaimer**: This model predicts risk likelihood only — NOT a medical diagnosis.